# HistGradientBoosting baseline: raw vs log1p

`03_target_transformation.ipynb`과 동일한 train split, 전처리, 5-Fold KFold에서 `HistGradientBoostingRegressor(random_state=42)`의 raw target과 `log1p` target을 비교합니다.

이번 단계에서는 HGB 두 후보만 새로 학습하며 고정 test는 예측·평가·후보 선택에 사용하지 않습니다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)


pandas: 3.0.5
NumPy: 2.5.3
scikit-learn: 1.9.1


## Method

중복 제거와 차량 제원 숫자 변환, 가격 구간 기반 80:20 split을 기존과 동일하게 재현합니다. 각 fold의 training subset에서만 결측 대체, missing indicator, `name_prefix_2`, One-Hot Encoding을 fit하고 validation에는 transform만 적용합니다.

HGB의 hyperparameter는 조정하지 않습니다. raw/log1p 모두 역변환 후 원래 가격 단위에서 MAE·RMSE·R²를 계산합니다.


In [2]:
data_path = Path("../data/raw/Car details v3.csv")
df = pd.read_csv(data_path)
df_clean = df.drop_duplicates().copy()
df_preprocessed = df_clean.copy()

for column in ["mileage", "engine", "max_power"]:
    df_preprocessed[column] = pd.to_numeric(
        df_preprocessed[column].str.extract(r"^([0-9]+(?:\.[0-9]+)?)", expand=False),
        errors="coerce",
    )

df_preprocessed.loc[df_preprocessed["mileage"].eq(0), "mileage"] = pd.NA
df_preprocessed.loc[df_preprocessed["max_power"].eq(0), "max_power"] = pd.NA

torque_text = df_preprocessed["torque"].str.strip()
torque_lower = torque_text.str.lower()
torque_first_number = pd.to_numeric(
    torque_text.str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False),
    errors="coerce",
)
nm_mask = torque_lower.str.contains("nm", regex=False, na=False)
kgm_mask = torque_lower.str.contains("kgm", regex=False, na=False)
nm_only_mask = nm_mask & ~kgm_mask
kgm_only_mask = kgm_mask & ~nm_mask
both_mask = nm_mask & kgm_mask
no_unit_mask = ~nm_mask & ~kgm_mask & torque_text.notna()

torque_nm = pd.Series(np.nan, index=df_preprocessed.index, dtype="float64")
torque_nm.loc[nm_only_mask] = torque_first_number.loc[nm_only_mask]
torque_nm.loc[kgm_only_mask & torque_first_number.lt(50)] = (
    torque_first_number.loc[kgm_only_mask & torque_first_number.lt(50)] * 9.80665
)
torque_nm.loc[both_mask] = torque_first_number.loc[both_mask]
torque_nm.loc[torque_text.eq("789Nm@ 2250rpm")] = np.nan

bmw_kgm_strings = [
    "51@ 1,750-3,000(kgm@ rpm)",
    "53@ 2,000-2,750(kgm@ rpm)",
]
bmw_kgm_mask = torque_text.isin(bmw_kgm_strings)
torque_nm.loc[bmw_kgm_mask] = torque_first_number.loc[bmw_kgm_mask] * 9.80665
remaining_large_kgm_mask = kgm_only_mask & torque_first_number.ge(50) & ~bmw_kgm_mask
torque_nm.loc[remaining_large_kgm_mask] = torque_first_number.loc[remaining_large_kgm_mask]
torque_nm.loc[no_unit_mask] = torque_first_number.loc[no_unit_mask]
df_preprocessed["torque"] = torque_nm

price_bins = pd.cut(
    df_preprocessed["selling_price"],
    bins=[0, 200000, 400000, 600000, 1000000, float("inf")],
    labels=["0~199,999", "200,000~399,999", "400,000~599,999", "600,000~999,999", "1,000,000 이상"],
    right=False,
    include_lowest=True,
)
X = df_preprocessed.drop(columns=["selling_price"]).copy()
y = df_preprocessed["selling_price"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=1385, random_state=42, stratify=price_bins
)

assert df_preprocessed.shape == (6926, 13)
assert X_train.shape == (5541, 12) and y_train.shape == (5541,)
assert X_train.index.equals(y_train.index)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("고정 test는 분할만 재현했으며 이번 비교에 사용하지 않음:", X_test.shape, y_test.shape)


X_train: (5541, 12)
y_train: (5541,)
고정 test는 분할만 재현했으며 이번 비교에 사용하지 않음: (1385, 12) (1385,)


In [3]:
def make_name_prefix_2(frame):
    result = frame.copy()
    result["name_prefix_2"] = result["name"].str.split().str[:2].str.join(" ")
    return result.drop(columns=["name"])


name_prefix_transformer = FunctionTransformer(make_name_prefix_2, validate=False)
base_numeric_cols = ["year", "km_driven"]
continuous_spec_cols = ["mileage", "engine", "max_power", "torque"]
seats_col = ["seats"]
categorical_cols = ["name_prefix_2", "fuel", "seller_type", "transmission", "owner"]

hgb_preprocessor = ColumnTransformer(
    transformers=[
        ("base_numeric", "passthrough", base_numeric_cols),
        ("continuous_specs", SimpleImputer(strategy="median"), continuous_spec_cols),
        ("continuous_indicators", MissingIndicator(features="all"), continuous_spec_cols),
        ("seats", SimpleImputer(strategy="most_frequent"), seats_col),
        ("seats_indicator", MissingIndicator(features="all"), seats_col),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

hgb_pipeline = Pipeline([
    ("preprocessing", Pipeline([
        ("name_prefix", clone(name_prefix_transformer)),
        ("preprocessor", clone(hgb_preprocessor)),
    ])),
    ("model", HistGradientBoostingRegressor(random_state=42)),
])
print("HGB Pipeline 준비 완료")
print("model:", hgb_pipeline.named_steps["model"])


HGB Pipeline 준비 완료
model: HistGradientBoostingRegressor(random_state=42)


In [4]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
fold_splits = list(cv.split(X_train))
candidates = [
    ("HistGradientBoosting", "raw", hgb_pipeline),
    ("HistGradientBoosting", "log1p", TransformedTargetRegressor(
        regressor=clone(hgb_pipeline), func=np.log1p, inverse_func=np.expm1
    )),
]
print("CV:", cv)
print("새로 학습할 후보 수:", len(candidates))


CV: KFold(n_splits=5, random_state=42, shuffle=True)
새로 학습할 후보 수: 2


In [5]:
def evaluate_candidate(model_name, target_representation, estimator):
    oof_prediction = pd.Series(np.nan, index=y_train.index, dtype="float64")
    validation_coverage = pd.Series(0, index=y_train.index, dtype="int64")
    fold_rows = []

    for fold, (train_positions, validation_positions) in enumerate(fold_splits, start=1):
        fold_estimator = clone(estimator)
        fold_estimator.fit(X_train.iloc[train_positions], y_train.iloc[train_positions])
        prediction = fold_estimator.predict(X_train.iloc[validation_positions])
        actual = y_train.iloc[validation_positions]
        oof_prediction.iloc[validation_positions] = prediction
        validation_coverage.iloc[validation_positions] += 1
        fold_rows.append({
            "model": model_name,
            "target_representation": target_representation,
            "fold": fold,
            "mae": mean_absolute_error(actual, prediction),
            "rmse": root_mean_squared_error(actual, prediction),
            "r2": r2_score(actual, prediction),
        })

    assert oof_prediction.isna().sum() == 0
    assert validation_coverage.eq(1).all()
    return pd.DataFrame(fold_rows), oof_prediction, validation_coverage


fold_results = []
oof_predictions = {}
coverage_results = {}
for model_name, target_representation, estimator in candidates:
    key = f"HGB_{target_representation}"
    candidate_folds, candidate_oof, candidate_coverage = evaluate_candidate(
        model_name, target_representation, estimator
    )
    fold_results.append(candidate_folds)
    oof_predictions[key] = candidate_oof
    coverage_results[key] = candidate_coverage
    print("완료:", key)

hgb_fold_metrics = pd.concat(fold_results, ignore_index=True)
print("candidate-fold 결과 수:", len(hgb_fold_metrics))


완료: HGB_raw
완료: HGB_log1p
candidate-fold 결과 수: 10


In [6]:
hgb_summary_rows = []
for target_representation, group in hgb_fold_metrics.groupby("target_representation", sort=False):
    key = f"HGB_{target_representation}"
    predictions = oof_predictions[key]
    negative_count = int((predictions < 0).sum())
    hgb_summary_rows.append({
        "model": "HistGradientBoosting",
        "target_representation": target_representation,
        "cv_mae_mean": group["mae"].mean(),
        "cv_mae_std": group["mae"].std(ddof=0),
        "cv_rmse_mean": group["rmse"].mean(),
        "cv_rmse_std": group["rmse"].std(ddof=0),
        "cv_r2_mean": group["r2"].mean(),
        "cv_r2_std": group["r2"].std(ddof=0),
        "oof_negative_prediction_count": negative_count,
        "oof_negative_prediction_pct": negative_count / len(predictions) * 100,
    })

hgb_summary = pd.DataFrame(hgb_summary_rows)
print("HGB summary")
print(hgb_summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("\nHGB fold metrics")
print(hgb_fold_metrics.to_string(index=False, float_format=lambda value: f"{value:.6f}"))


HGB summary
               model target_representation  cv_mae_mean  cv_mae_std  cv_rmse_mean  cv_rmse_std  cv_r2_mean  cv_r2_std  oof_negative_prediction_count  oof_negative_prediction_pct
HistGradientBoosting                   raw 80754.415602 3755.261122 180995.704754 32243.514499    0.878571   0.035625                              7                     0.126331
HistGradientBoosting                 log1p 79078.826804 3904.748659 183795.907030 29047.864094    0.876332   0.027903                              0                     0.000000

HGB fold metrics
               model target_representation  fold          mae          rmse       r2
HistGradientBoosting                   raw     1 76074.379045 150227.141506 0.880243
HistGradientBoosting                   raw     2 82517.659108 187007.729073 0.891725
HistGradientBoosting                   raw     3 86834.609966 240700.929473 0.813419
HistGradientBoosting                   raw     4 77844.964365 159051.851963 0.921885
HistGradien

In [7]:
hgb_raw = hgb_summary.query("target_representation == 'raw'").iloc[0]
hgb_log = hgb_summary.query("target_representation == 'log1p'").iloc[0]
hgb_raw_folds = hgb_fold_metrics.query("target_representation == 'raw'").sort_values("fold")
hgb_log_folds = hgb_fold_metrics.query("target_representation == 'log1p'").sort_values("fold")

hgb_target_change = pd.Series({
    "mae_raw_minus_log": hgb_raw["cv_mae_mean"] - hgb_log["cv_mae_mean"],
    "mae_change_pct": (hgb_raw["cv_mae_mean"] - hgb_log["cv_mae_mean"]) / hgb_raw["cv_mae_mean"] * 100,
    "rmse_raw_minus_log": hgb_raw["cv_rmse_mean"] - hgb_log["cv_rmse_mean"],
    "rmse_change_pct": (hgb_raw["cv_rmse_mean"] - hgb_log["cv_rmse_mean"]) / hgb_raw["cv_rmse_mean"] * 100,
    "r2_log_minus_raw": hgb_log["cv_r2_mean"] - hgb_raw["cv_r2_mean"],
    "mae_improved_folds": int((hgb_log_folds["mae"].to_numpy() < hgb_raw_folds["mae"].to_numpy()).sum()),
    "rmse_improved_folds": int((hgb_log_folds["rmse"].to_numpy() < hgb_raw_folds["rmse"].to_numpy()).sum()),
})
print(hgb_target_change.to_string(float_format=lambda value: f"{value:.6f}"))


mae_raw_minus_log      1675.588798
mae_change_pct            2.074919
rmse_raw_minus_log    -2800.202276
rmse_change_pct          -1.547110
r2_log_minus_raw         -0.002238
mae_improved_folds        4.000000
rmse_improved_folds       3.000000


In [8]:
confirmed_results = pd.DataFrame([
    {"model": "Ridge", "target_representation": "raw", "cv_mae_mean": 124406.465627, "cv_mae_std": 3727.913627, "cv_rmse_mean": 245443.803774, "cv_rmse_std": 31806.307755, "cv_r2_mean": 0.780487, "cv_r2_std": 0.033710, "oof_negative_prediction_count": 234},
    {"model": "Ridge", "target_representation": "log1p", "cv_mae_mean": 83323.839206, "cv_mae_std": 3516.986113, "cv_rmse_mean": 189720.989487, "cv_rmse_std": 15915.812404, "cv_r2_mean": 0.867377, "cv_r2_std": 0.020256, "oof_negative_prediction_count": 0},
    {"model": "RandomForest", "target_representation": "raw", "cv_mae_mean": 78457.889024, "cv_mae_std": 3867.836060, "cv_rmse_mean": 164896.704467, "cv_rmse_std": 28340.606040, "cv_r2_mean": 0.898948, "cv_r2_std": 0.027368, "oof_negative_prediction_count": 0},
    {"model": "RandomForest", "target_representation": "log1p", "cv_mae_mean": 78161.849416, "cv_mae_std": 4130.050820, "cv_rmse_mean": 171738.880502, "cv_rmse_std": 31731.176789, "cv_r2_mean": 0.889622, "cv_r2_std": 0.036100, "oof_negative_prediction_count": 0},
])

six_candidate_summary = pd.concat([
    confirmed_results,
    hgb_summary[confirmed_results.columns],
], ignore_index=True)
six_candidate_summary["candidate"] = (
    six_candidate_summary["model"] + " " + six_candidate_summary["target_representation"]
)
six_candidate_summary["mae_rank"] = six_candidate_summary["cv_mae_mean"].rank(method="min", ascending=True).astype(int)
six_candidate_summary["rmse_rank"] = six_candidate_summary["cv_rmse_mean"].rank(method="min", ascending=True).astype(int)
six_candidate_summary["r2_rank"] = six_candidate_summary["cv_r2_mean"].rank(method="min", ascending=False).astype(int)

print(six_candidate_summary[[
    "candidate", "cv_mae_mean", "cv_mae_std", "cv_rmse_mean", "cv_rmse_std",
    "cv_r2_mean", "cv_r2_std", "oof_negative_prediction_count",
    "mae_rank", "rmse_rank", "r2_rank",
]].to_string(index=False, float_format=lambda value: f"{value:.6f}"))

for metric, rank_column in [("MAE", "mae_rank"), ("RMSE", "rmse_rank"), ("R²", "r2_rank")]:
    ranking = six_candidate_summary.sort_values(rank_column)["candidate"].tolist()
    print(f"{metric} rank:", " > ".join(ranking))


                 candidate   cv_mae_mean  cv_mae_std  cv_rmse_mean  cv_rmse_std  cv_r2_mean  cv_r2_std  oof_negative_prediction_count  mae_rank  rmse_rank  r2_rank
                 Ridge raw 124406.465627 3727.913627 245443.803774 31806.307755    0.780487   0.033710                            234         6          6        6
               Ridge log1p  83323.839206 3516.986113 189720.989487 15915.812404    0.867377   0.020256                              0         5          5        5
          RandomForest raw  78457.889024 3867.836060 164896.704467 28340.606040    0.898948   0.027368                              0         2          1        1
        RandomForest log1p  78161.849416 4130.050820 171738.880502 31731.176789    0.889622   0.036100                              0         1          2        2
  HistGradientBoosting raw  80754.415602 3755.261122 180995.704754 32243.514499    0.878571   0.035625                              7         4          3        3
HistGradientBoos

In [9]:
rf_raw = confirmed_results.query(
    "model == 'RandomForest' and target_representation == 'raw'"
).iloc[0]
rf_vs_hgb_raw = pd.Series({
    "RF_raw_MAE": rf_raw["cv_mae_mean"],
    "HGB_raw_MAE": hgb_raw["cv_mae_mean"],
    "HGB_minus_RF_MAE": hgb_raw["cv_mae_mean"] - rf_raw["cv_mae_mean"],
    "RF_raw_RMSE": rf_raw["cv_rmse_mean"],
    "HGB_raw_RMSE": hgb_raw["cv_rmse_mean"],
    "HGB_minus_RF_RMSE": hgb_raw["cv_rmse_mean"] - rf_raw["cv_rmse_mean"],
    "RF_raw_R2": rf_raw["cv_r2_mean"],
    "HGB_raw_R2": hgb_raw["cv_r2_mean"],
    "HGB_minus_RF_R2": hgb_raw["cv_r2_mean"] - rf_raw["cv_r2_mean"],
})

oof_integrity = pd.DataFrame({
    key: {
        "total": len(predictions),
        "missing": int(predictions.isna().sum()),
        "duplicate_validation_coverage": int((coverage_results[key] > 1).sum()),
        "uncovered": int((coverage_results[key] == 0).sum()),
    }
    for key, predictions in oof_predictions.items()
}).T

test_usage = pd.Series({
    "test_prediction": False,
    "test_metric": False,
    "test_based_selection": False,
})
print("RF raw vs HGB raw")
print(rf_vs_hgb_raw.to_string(float_format=lambda value: f"{value:.6f}"))
print("\nOOF integrity")
print(oof_integrity.to_string())
print("\ntest usage")
print(test_usage.to_string())


RF raw vs HGB raw
RF_raw_MAE           78457.889024
HGB_raw_MAE          80754.415602
HGB_minus_RF_MAE      2296.526578
RF_raw_RMSE         164896.704467
HGB_raw_RMSE        180995.704754
HGB_minus_RF_RMSE    16099.000287
RF_raw_R2                0.898948
HGB_raw_R2               0.878571
HGB_minus_RF_R2         -0.020377

OOF integrity
           total  missing  duplicate_validation_coverage  uncovered
HGB_raw     5541        0                              0          0
HGB_log1p   5541        0                              0          0

test usage
test_prediction         False
test_metric             False
test_based_selection    False


## Takeaways

- HGB log1p는 raw 대비 MAE가 **개선**, RMSE가 **악화**, R²가 **악화**했습니다.
- MAE 개선 fold는 **4/5**, RMSE 개선 fold는 **3/5**입니다.
- 이 결과는 HGB 기본 설정 한 번의 train 내부 CV 비교이며 hyperparameter tuning이나 고정 test 평가는 수행하지 않았습니다.
